In [1]:
sequence = "maktab"
print(f"Sequence: {sequence}")

# sequence nomli string o'zgaruvchisini aniqlaydi va unga "maktab" qiymatini yuklaydi.
# Keyin, sequence o'zgaruvchisining qiymatini formatlangan chiqish uchun f-string yordamida konsolga chop etadi.

Sequence: maktab


In [2]:
chars = sorted(list(set(sequence)))
# set(sequence): sequence dagi harflarning noyob to'plamini (set) yaratadi, ya'ni takrorlanuvchi harflarni olib tashlaydi.
# list(...): To'plamni (set) ro'yxatga (list) aylantiradi.
# sorted(...): Ro'yxatdagi harflarni alifbo tartibida saralaydi.
print(f"Unique characters: {chars}")

Unique characters: ['a', 'b', 'k', 'm', 't']


In [3]:
char2idx = {char: i for i, char in enumerate(chars)}
idx2char = {i: char for i, char in enumerate(chars)}
# char2idx: Har bir noyob harfga uning tartib raqamini (indeksini) belgilaydi.
# Misol uchun, 'a' harfi 0-indeksga, 'b' harfi 1-indeksga va hokazo o'tkaziladi.
# idx2char: Yuqoridagi jarayonning teskarisini qiladi, ya'ni indeksdan unga mos keladigan harfni
# topish imkonini beradi. Misol uchun, 0-indeks 'a' harfiga, 1-indeks 'b' harfiga va hokazo o'tkaziladi.

print(f"Character to index mapping: {char2idx}")
print(f"Index to character mapping: {idx2char}")

Character to index mapping: {'a': 0, 'b': 1, 'k': 2, 'm': 3, 't': 4}
Index to character mapping: {0: 'a', 1: 'b', 2: 'k', 3: 'm', 4: 't'}


In [4]:
x_data = [char2idx[char] for char in sequence[:-1]]
y_data = [char2idx[char] for char in sequence[1:]]
# x_data = [char2idx[char] for char in sequence[:-1]]: sequence so'zining oxirgi harfidan tashqari barcha
# harflarni oladi va ularni char2idx lug'ati yordamida indekslarga aylantiradi. Bu modelga keyingi harfni
# bashorat qilish uchun joriy harflar ketma-ketligini beradi.
# y_data = [char2idx[char] for char in sequence[1:]]: sequence so'zining birinchi harfidan tashqari
# barcha harflarni oladi va ularni indekslarga aylantiradi. Bu x_data dagi harflarning keyingi
# bashorati bo'lishi kerak bo'lgan haqiqiy qiymatlarni ifodalaydi.

print(f"x_data (indexed): {x_data}")
print(f"y_data (indexed): {y_data}")

x_data (indexed): [3, 0, 2, 4, 0]
y_data (indexed): [0, 2, 4, 0, 1]


In [5]:
import torch

x_data_tensor = torch.tensor(x_data).unsqueeze(0) # Add batch dimension
y_data_tensor = torch.tensor(y_data).unsqueeze(0) # Add batch dimension

print(f"x_data_tensor shape: {x_data_tensor.shape}")
print(f"y_data_tensor shape: {y_data_tensor.shape}")
print(f"x_data_tensor:\n{x_data_tensor}")
print(f"y_data_tensor:\n{y_data_tensor}")

# import torch: PyTorch kutubxonasini import qiladi.
# x_data_tensor = torch.tensor(x_data).unsqueeze(0): x_data ro'yxatini PyTorch tensoriga
# o'tkazadi va unsqueeze(0) yordamida unga 'batch' o'lchamini qo'shadi. Bu neyron
# tarmoqlarning kirish ma'lumotlari uchun standart shakl hisoblanadi.
# y_data_tensor = torch.tensor(y_data).unsqueeze(0): Xuddi shunday,
# y_data ro'yxatini PyTorch tensoriga aylantiradi va unga 'batch' o'lchamini qo'shadi.
# Natijada, x_data_tensor va y_data_tensor modelni o'qitish uchun tayyor bo'lgan mos shakldagi tensorlarga aylanadi.



x_data_tensor shape: torch.Size([1, 5])
y_data_tensor shape: torch.Size([1, 5])
x_data_tensor:
tensor([[3, 0, 2, 4, 0]])
y_data_tensor:
tensor([[0, 2, 4, 0, 1]])


In [6]:
import torch.nn as nn
import torch.nn.functional as F

class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(CharRNN, self).__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.rnn = nn.RNN(input_size=vocab_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(in_features=hidden_size, out_features=vocab_size)

    def forward(self, x, hidden):
        # Convert input x to one-hot encoded vectors
        x_one_hot = F.one_hot(x, num_classes=self.vocab_size).float()

        # Pass through RNN layer
        output, hidden = self.rnn(x_one_hot, hidden)

        # Pass through linear layer
        output = self.fc(output)

        return output, hidden

print("CharRNN class defined successfully.")

# import torch.nn as nn va import torch.nn.functional as F: Neyron tarmoq qatlamlari va funksiyalari uchun kerakli kutubxonalarni import qiladi.
# CharRNN klassi: nn.Module dan meros oladi, ya'ni bu PyTorch ning asosiy neyron tarmoq modellaridan biri.
# __init__(self, vocab_size, hidden_size) metodi: Modelning qatlamlarini (RNN va chiziqli qatlam) ishga tushiradi:
# vocab_size: Lug'atdagi noyob harflar soni (kirish va chiqish o'lchamlari uchun).
# hidden_size: RNN ning ichki holatining o'lchami.
# self.rnn = nn.RNN(...): Kirish ma'lumotlarini qayta ishlaydigan qaytariluvchi neyron tarmoq qatlami.
# self.fc = nn.Linear(...): RNN chiqishini yakuniy bashoratlarga aylantiradigan chiziqli qatlam.
# forward(self, x, hidden) metodi: Modelning ma'lumotlarni qanday qayta ishlashini belgilaydi:
# x_one_hot = F.one_hot(x, num_classes=self.vocab_size).float(): Kirish harf indekslarini bir-issiq
# (one-hot) vektorlarga aylantiradi. Bu RNN uchun kirish formatidir.
# output, hidden = self.rnn(x_one_hot, hidden): Bir-issiq vektorlarni RNN qatlamiga o'tkazadi va chiqish hamda yangi yashirin holatni oladi.
# output = self.fc(output): RNN chiqishini chiziqli qatlamdan o'tkazib, har bir harf uchun ehtimolliklar (yoki logitlar) ni hosil qiladi.
# return output, hidden: Yakuniy chiqishni va yangi yashirin holatni qaytaradi.

CharRNN class defined successfully.


In [7]:
hidden_size = 10
lr = 0.02
epochs = 200

vocab_size = len(chars)

model = CharRNN(vocab_size, hidden_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print(f"Model initialized:\n{model}")
print(f"Optimizer: {optimizer}")
print(f"Criterion: {criterion}")

# hidden_size = 10, lr = 0.02, epochs = 200: hidden_size (RNN ichki holatining o'lchami),
#lr (o'rganish tezligi) va epochs (o'qitish sikllari soni) kabi giperparametrlar o'rnatiladi.
# vocab_size = len(chars): Lug'atdagi noyob harflar soni hisoblanadi, bu modelning kirish va chiqish o'lchamlari uchun muhim.
# model = CharRNN(vocab_size, hidden_size): Yuqorida aniqlangan CharRNN klassidan model yaratiladi.
# criterion = nn.CrossEntropyLoss(): Yo'qotish funksiyasi sifatida CrossEntropyLoss tanlanadi,
# bu tasniflash (classification) vazifalari uchun keng qo'llaniladi.
# optimizer = torch.optim.Adam(model.parameters(), lr=lr): Adam optimizatori modelning
# parametrlarini lr o'rganish tezligi bilan yangilash uchun sozlanadi.

Model initialized:
CharRNN(
  (rnn): RNN(5, 10, batch_first=True)
  (fc): Linear(in_features=10, out_features=5, bias=True)
)
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.02
    maximize: False
    weight_decay: 0
)
Criterion: CrossEntropyLoss()


In [8]:
for epoch in range(epochs):
    hidden = torch.zeros(1, 1, hidden_size)

    optimizer.zero_grad()

    output, hidden = model(x_data_tensor, hidden)

    loss = criterion(output.view(-1, vocab_size), y_data_tensor.view(-1))

    loss.backward()

    optimizer.step()

    hidden = hidden.detach()

    if (epoch + 1) % 20 == 0:
        print(f'Epoch: {epoch + 1}, Loss: {loss.item():.4f}')

        _, pred_indices = torch.max(output, dim=2)
        pred_seq = ''.join([idx2char[idx.item()] for idx in pred_indices.squeeze(0)])
        print(f'Predicted sequence: {pred_seq}')

# for epoch in range(epochs): Modelni belgilangan epochs (200) soni bo'yicha o'qitish siklini boshlaydi.
# hidden = torch.zeros(1, 1, hidden_size): Har bir epoxa boshida RNN ning yashirin holatini (hidden state)
# nolga o'rnatadi. Bu modelni har bir yangi ma'lumot ketma-ketligi uchun toza holatda boshlashga yordam beradi.
# optimizer.zero_grad(): Oldingi iteratsiyadan qolgan gradientlarni tozalaydi, bu yangi iteratsiyada faqat joriy
# hisob-kitoblar asosida gradientlarni yig'ish uchun muhim.
# output, hidden = model(x_data_tensor, hidden): Modelga kiritiladigan ma'lumot (x_data_tensor) va yashirin
# holatni yuboradi, modeldan chiqish (output) va yangi yashirin holatni oladi.
# loss = criterion(output.view(-1, vocab_size), y_data_tensor.view(-1)): Modelning chiqishi (output) va
# haqiqiy natija (y_data_tensor) o'rtasidagi farqni (yo'qotishni) hisoblaydi.
# loss.backward(): Yo'qotish funksiyasi orqali modelning barcha o'rganiladigan parametrlariga nisbatan gradientlarni hisoblaydi (orqaga tarqatish).
# optimizer.step(): Hisoblangan gradientlar yordamida model parametrlarini yangilaydi (modelni o'qitadi).
# hidden = hidden.detach(): Yashirin holatni hisoblash grafigidan ajratib oladi, bu oldingi
# iteratsiyalardagi hisob-kitoblarga bog'lanib qolishning oldini oladi va xotira muammolarini kamaytiradi.
# if (epoch + 1) % 20 == 0: Har 20 epoxada o'qitish holatini konsolga chiqaradi:
# print(f'Epoch: {epoch + 1}, Loss: {loss.item():.4f}'): Joriy epoxa raqami va yo'qotish qiymatini ko'rsatadi.
# _, pred_indices = torch.max(output, dim=2): Modelning chiqishidan eng yuqori ehtimollikka ega bo'lgan harf indekslarini aniqlaydi.
# pred_seq = ''.join([idx2char[idx.item()] for idx in pred_indices.squeeze(0)]): Bashorat qilingan
# harf indekslarini idx2char lug'ati yordamida harflarga aylantirib, bashorat qilingan ketma-ketlikni hosil qiladi.
# print(f'Predicted sequence: {pred_seq}'): Bashorat qilingan ketma-ketlikni ko'rsatadi.


Epoch: 20, Loss: 0.1981
Predicted sequence: aktab
Epoch: 40, Loss: 0.0214
Predicted sequence: aktab
Epoch: 60, Loss: 0.0079
Predicted sequence: aktab
Epoch: 80, Loss: 0.0049
Predicted sequence: aktab
Epoch: 100, Loss: 0.0037
Predicted sequence: aktab
Epoch: 120, Loss: 0.0029
Predicted sequence: aktab
Epoch: 140, Loss: 0.0023
Predicted sequence: aktab
Epoch: 160, Loss: 0.0020
Predicted sequence: aktab
Epoch: 180, Loss: 0.0017
Predicted sequence: aktab
Epoch: 200, Loss: 0.0014
Predicted sequence: aktab


Biz bajargan barcha ishlarning asosiy maqsadi — 'maktab' so'zining keyingi harflarini bashorat qilishni o'rganadigan oddiy takrorlanuvchi neyron tarmoq (RNN) modelini yaratish va o'qitish edi.

Qisqacha aytganda:

Ma'lumotlarni tayyorlash: 'maktab' so'zidagi harflarni sonli indekslarga aylantirdik va modelga kirish hamda chiqish ma'lumotlarini (x_data va y_data) shakllantirdik.

Modelni yaratish: Harf ketma-ketliklarini qayta ishlashga qodir bo'lgan CharRNN nomli neyron tarmoq modelini tuzdik.

Modelni o'qitish: Tayyorlangan ma'lumotlar yordamida modelni o'qitdik, bunda model 'maktab' so'zidagi harflar orasidagi ketma-ketlik qoidasini o'rganishga harakat qildi.

Natijada, bizning modelimiz 'maktab' so'zidagi harflar ketma-ketligini samarali tarzda o'rganib, keyingi harflarni to'g'ri bashorat qila oldi. Bu RNN larning matn bashorati kabi vazifalarda qanday ishlashini ko'rsatuvchi kichik bir misol.

